# Statistical Analysis
## OHLCV Price Statistics

### Objectives:
- Distribution analysis of prices and returns
- Skewness and kurtosis examination
- Outlier detection using IQR and Z-score methods
- Rolling statistics analysis
- Volatility and return distribution patterns

In [1]:
# ====================================================================
# 📦 IMPORTS AND SETUP
# ====================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# ====================================================================
# 🔧 PATH CONFIGURATION - Find utils folder
# ====================================================================

# Get notebook location
notebook_dir = Path(os.getcwd()).resolve()

def find_ohlcv_root(start_path):
    current = start_path
    for _ in range(5):
        if (current / 'utils').exists() and (current / 'utils' / 'data_loader.py').exists():
            return current
        current = current.parent
    return None

# Find ohlcv root
ohlcv_root = find_ohlcv_root(notebook_dir)

if ohlcv_root:
    utils_path = ohlcv_root / 'utils'
    if str(utils_path) not in sys.path:
        sys.path.insert(0, str(utils_path))
    print(f"✅ OHLCV root: {ohlcv_root}")
    print(f"✅ Utils path: {utils_path}")
else:
    print("⚠️  Could not find ohlcv root. Trying relative path...")
    for rel_path in ['../../utils', '../../../utils', '../../../../utils']:
        test_path = (notebook_dir / rel_path).resolve()
        if test_path.exists() and (test_path / 'data_loader.py').exists():
            if str(test_path) not in sys.path:
                sys.path.insert(0, str(test_path))
            print(f"✅ Found utils at: {test_path}")
            break

# ====================================================================
# 📦 IMPORT UTILITIES
# ====================================================================

try:
    from data_loader import DataLoader
    from visualizations import TradingVisualizer
    from trading_helpers import TradingHelpers
    print("✅ All utilities imported successfully!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("⚠️  Please ensure utils folder exists with required files.")
    raise

print("\n" + "=" * 60)
print("✅ SETUP COMPLETE")
print("=" * 60)

✅ OHLCV root: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\ohlcv
✅ Utils path: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\ohlcv\utils
✅ All utilities imported successfully!

✅ SETUP COMPLETE


In [2]:
# Load data
loader = DataLoader()
df = loader.load_from_csv()

if df.empty:
    df = loader.load_from_db()

print(f"✅ Data loaded: {len(df)} rows")

# Calculate returns
df['returns'] = df['close'].pct_change() * 100
df['log_returns'] = np.log(df['close'] / df['close'].shift(1)) * 100
df['range'] = df['high'] - df['low']
df['range_pct'] = (df['high'] - df['low']) / df['close'] * 100

# Price changes
df['price_change'] = df['close'] - df['open']
df['price_change_pct'] = (df['close'] - df['open']) / df['open'] * 100

df.head()

⚠️  CSV file not found: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\data\Eth_OHLCV.csv
⚠️  Database not found: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\data\ETH.db
✅ Data loaded: 0 rows


KeyError: 'close'

In [ ]:
# Distribution analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Price distribution
df['close'].hist(bins=50, ax=axes[0, 0], edgecolor='black')
axes[0, 0].set_title('ETH Price Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency')

# Returns distribution
df['returns'].dropna().hist(bins=50, ax=axes[0, 1], edgecolor='black', color='orange')
axes[0, 1].set_title('Returns Distribution', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Returns (%)')
axes[0, 1].set_ylabel('Frequency')

# QQ plot for returns
from scipy import stats
stats.probplot(df['returns'].dropna(), dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot for Returns', fontsize=14, fontweight='bold')

# Box plot for returns
df['returns'].dropna().boxplot(ax=axes[1, 1])
axes[1, 1].set_title('Returns Box Plot', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Returns (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Statistical metrics
print("=" * 60)
print("📊 PRICE STATISTICS")
print("=" * 60)
price_stats = df[['open', 'high', 'low', 'close', 'volume']].describe()
print(price_stats)

print("\n" + "=" * 60)
print("📊 RETURNS STATISTICS")
print("=" * 60)
returns_stats = df[['returns', 'log_returns']].describe()
print(returns_stats)

print("\n" + "=" * 60)
print("📊 SKEWNESS & KURTOSIS")
print("=" * 60)
for col in ['close', 'returns', 'log_returns']:
    if col in df.columns:
        data = df[col].dropna()
        skew = data.skew()
        kurt = data.kurtosis()
        print(f"{col:15s} | Skewness: {skew:8.4f} | Kurtosis: {kurt:8.4f}")

In [ ]:
# Outlier detection
print("=" * 60)
print("🔍 OUTLIER DETECTION")
print("=" * 60)

# IQR method
def detect_outliers_iqr(data, multiplier=1.5):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    outliers = (data < lower_bound) | (data > upper_bound)
    return outliers, lower_bound, upper_bound

# Detect outliers in returns
for multiplier in [1.5, 2.0, 3.0]:
    outliers, lb, ub = detect_outliers_iqr(df['returns'].dropna(), multiplier)
    count = outliers.sum()
    pct = count / len(df['returns'].dropna()) * 100
    print(f"IQR (multiplier={multiplier}): {count} outliers ({pct:.2f}%)")
    print(f"  Bounds: [{lb:.2f}, {ub:.2f}]\n")

# Z-score method
from scipy import stats
returns = df['returns'].dropna()
z_scores = np.abs(stats.zscore(returns))

for threshold in [2, 2.5, 3]:
    outliers = z_scores > threshold
    count = outliers.sum()
    pct = count / len(returns) * 100
    print(f"Z-Score (threshold={threshold}): {count} outliers ({pct:.2f}%)")